# SynBioCrow 2.3 — DBTL Sequence-Backed Build V6
Uses a provenance-backed sabinene/valencene protein, designs a translation-preserving E. coli-preferred synthetic CDS, and automatically repairs forbidden restriction motifs by synonymous codon substitution before rerunning sequence QC. Full cassette assembly still requires verified regulatory-part sequences.


In [ ]:
import sys,subprocess,shutil,json,zipfile,hashlib,time
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_dbtl_build_v2")
NORM=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_literature_normalized/galaxy_literature_benchmark_normalized.json")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/dbtl_sequence_backed_build")
OUT.mkdir(parents=True,exist_ok=True)

def run(cmd,*,cwd=None,timeout=1200):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if p.returncode: raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.3",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=900)
assert NORM.is_file(),NORM

run([sys.executable,str(REPO/"scripts/run_dbtl_sequence_backed_build_2_3.py"),
     "--normalized-benchmark",str(NORM),
     "--output-dir",str(OUT)],cwd=REPO,timeout=600)

bundle=OUT/"SynBioCrow_2_3_DBTL_SEQUENCE_BACKED_BUILD_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*.json")): z.write(p,p.name)
    z.write(REPO/"scripts/run_dbtl_sequence_backed_build_2_3.py","run_dbtl_sequence_backed_build_2_3.py")
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
